# 第三步 · 去主题语料的重嵌入

**目的**：字面空间那边已经证明，剥掉 605 个人名、地名、品牌、俚语、英文词条之后，
歌手身份信号几乎不变（0.450 → 0.444）。语义那一臂（BGE-M3）没法用同样的办法处理——
向量是一次性在 GPU 上算出来的。要回答"BGE-M3 保留的身份是不是靠名字"，得把剥完词的
语料**重新嵌一遍**，再在同一套协议下打分。

**和第二步完全一样的流程**，只是语料换成 `repaired_lyric_chunks_v2_neutralised.csv`
（25,026 段，169,396 个字符被剥掉，13,314 段有改动），输出目录换成 `embeddings-v2-neutralised/`。
内容摘要会核对，传错文件会停。

这个 notebook 里没有歌词，可以公开；语料从你的 Drive 读，**别放进共享文件夹**。

断线处理、GPU 门禁、分块落盘、逐位可复现——都和第二步相同。


## 1 · 确认 GPU

菜单 → 代码执行程序 → 更改运行时类型 → **T4 GPU**。

CPU 也能跑,但会慢到不现实(几十小时)。这一步基本必须有 GPU。

In [ ]:
!nvidia-smi -L || echo '没有 GPU —— 去 运行时→更改运行时类型 选 T4,否则这一步不现实'

## 2 · 装依赖

约 2–3 分钟,装完**不需要**重启运行时。

In [ ]:
!pip install -q FlagEmbedding==1.4.0 2>&1 | tail -2
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3 · 挂 Drive,放语料

把语料传到你的 Google Drive。两种形式都行:

```
repaired_lyric_chunks_v2.csv        22 MB
repaired_lyric_chunks_v2.csv.gz     9.2 MB,内容完全相同
```

放哪都行,下面会自己找:先看 `MyDrive/chinese-rap/`,再看 `MyDrive/` 根目录,还找不到就全盘搜。
校验是按解析后的行内容算的,跟你传的是压缩版还是原版无关。

**这个文件是私有的**,别放进任何共享文件夹。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gzip, io, pathlib

# Either form works. The gzipped copy exists because a 22 MB upload does not fit through
# every route; it is the same bytes, and the digest below is computed over parsed rows, so
# it does not care which one you uploaded.
NAMES = ['repaired_lyric_chunks_v2_neutralised.csv', 'repaired_lyric_chunks_v2_neutralised.csv.gz']
DIRS = [pathlib.Path('/content/drive/MyDrive/chinese-rap'),
        pathlib.Path('/content/drive/MyDrive')]

CORPUS = next((d / n for d in DIRS for n in NAMES if (d / n).is_file()), None)
if CORPUS is None:
    print('默认位置没找到,全盘搜索中...')
    CORPUS = next((p for n in NAMES
                   for p in pathlib.Path('/content/drive/MyDrive').rglob(n)), None)
if CORPUS is None:
    raise SystemExit('Drive 里找不到 repaired_lyric_chunks_v2_neutralised.csv 或 .csv.gz —— 先传上去')


def open_corpus():
    # A text handle on the corpus, gzipped or not.
    if CORPUS.suffix == '.gz':
        return io.TextIOWrapper(gzip.open(CORPUS, 'rb'), encoding='utf-8')
    return CORPUS.open(encoding='utf-8')


print('语料:', CORPUS, '(gzip)' if CORPUS.suffix == '.gz' else '')

OUT = CORPUS.parent / 'embeddings-v2-neutralised'
OUT.mkdir(exist_ok=True)
print('输出目录:', OUT)

## 4 · 校验语料是不是那一份

Drive 上传很容易传错版本,或者传成半截。这里按**内容摘要**核对——不是文件字节,
是 `(label, song_id, title, chunk_id, source_order, text)` 这个有序内容契约的 SHA-256,
和 `results/repaired-corpus-v2/analysis_summary.json` 里发布的那个比。

对不上就停,不往下跑。**在错的语料上烧三个小时 GPU 是没必要的。**

In [ ]:
import csv, hashlib, json
csv.field_size_limit(10 ** 9)

EXPECTED_DIGEST = '8b507a818e9fd9828b06725e71610483f72f0148318b0fe98bb39b4b4b2b9079'
EXPECTED_ROWS = 25026
EXPECTED_SONGS = 7391

rows = list(csv.DictReader(open_corpus()))

# same content contract as src/duplicate_control_v2.py::corpus_content_sha256
digest = hashlib.sha256()
for r in rows:
    canonical = json.dumps(
        [str(r['source_credit_label']), str(r['song_id']), str(r['song_title']),
         int(r['chunk_id']), int(r['source_order']), str(r['cleaned_text'])],
        ensure_ascii=False, separators=(',', ':'))
    digest.update((canonical + chr(10)).encode('utf-8'))
actual = digest.hexdigest()

songs = len({r['song_id'] for r in rows})
print('段落数 ', len(rows), '(应为', EXPECTED_ROWS, ')')
print('歌曲数 ', songs, '(应为', EXPECTED_SONGS, ')')
print('内容摘要', actual)
print('已发布  ', EXPECTED_DIGEST)

ok = (actual == EXPECTED_DIGEST and len(rows) == EXPECTED_ROWS and songs == EXPECTED_SONGS)
if not ok:
    raise SystemExit('语料和已发布的 corpus v2 对不上 —— 停在这里,先确认传的是哪一份')
print()
print('校验通过,这就是发布的那份 corpus v2')

## 5 · 下模型,核对权重

BGE-M3 约 2.3 GB,第一次下载几分钟。

契约里把权重的 SHA-256 钉死了。**不同的 BGE-M3 快照产出不同的向量**,所以下完先核对。
对不上不会停——但会记进 contract,后面读结果的人得知道这一点。

In [ ]:
import hashlib, pathlib
from huggingface_hub import snapshot_download

model_path = pathlib.Path(snapshot_download('BAAI/bge-m3'))
weights = next((model_path / n for n in ('pytorch_model.bin', 'model.safetensors')
                if (model_path / n).is_file()), None)

h = hashlib.sha256()
with open(weights, 'rb') as f:
    for b in iter(lambda: f.read(1 << 20), b''):
        h.update(b)
WEIGHTS_SHA256 = h.hexdigest()
print(weights.name, WEIGHTS_SHA256)

## 6 · 嵌入

**这一格可以反复跑。** 重连之后走「全部运行」,它会自己跳过已完成的分块。

开跑前会先验一遍已有的分块:能加载、行数对得上才算数。写了一半的会被删掉重算——
运行时中途死掉会留下截断的 .npy,光看文件在不在会把它当成已完成,那个错误要到几小时后
合并时才炸出来。

- 分块 512 段,每块存一次盘
- 按语料顺序走,不按长度排序 —— 续跑和一次跑完逐位相同
- `max_length=2048` 沿用契约;它是截断上限,不是把每段都补到 2048

进度会打出来。T4 上整体大约 1–3 小时,取决于卡的状态。

In [ ]:
import numpy as np, time
from FlagEmbedding import BGEM3FlagModel

BLOCK = 512
BATCH = 8
MAX_LENGTH = 2048
USE_FP16 = torch.cuda.is_available()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# A CPU runtime must not start, and must never continue, this run.
#
# The checkpoints carry no precision marker -- every block is cast to float32 before it is
# written, so a cpu/fp32 block and a cuda/fp16 block are indistinguishable on disk. Colab's
# usual failure for a job this long is a GPU-quota cutoff, and `!nvidia-smi` above only
# prints; a `!` line cannot fail a cell. Without this gate a reconnect onto a CPU runtime
# quietly appends fp32 blocks to an fp16 set, and the merged matrix carries the precision
# boundary that this entire notebook exists to remove -- undetectably, for the reason step 1
# established. Waiting for a T4 is the cheap option.
if not torch.cuda.is_available():
    raise SystemExit(chr(10).join([
        '没有 GPU,停在这里。',
        '  断点是 GPU 算的,在 CPU 上接着跑会把两种精度混进同一个向量集,事后分辨不出来。',
        '  而且 CPU 上跑完要几十小时,超过 Colab 单次会话上限,这个 run 永远跑不完。',
        '  去 运行时 → 更改运行时类型 选 T4;拿不到就等配额恢复再来。',
    ]))

texts = [r['cleaned_text'] for r in rows]
n_blocks = (len(texts) + BLOCK - 1) // BLOCK
ckpt_dir = OUT / 'blocks'
ckpt_dir.mkdir(exist_ok=True)

# Pin the configuration to the checkpoint directory on the first run and refuse to resume
# under a different one. Row count alone cannot tell these apart, so it has to be recorded.
RUN_CONFIG = {
    'device': DEVICE,
    'use_fp16': USE_FP16,
    'gpu': torch.cuda.get_device_name(0),
    'max_length': MAX_LENGTH,
    'block_size': BLOCK,
    'weights_sha256': WEIGHTS_SHA256,
    'corpus_content_sha256': EXPECTED_DIGEST,
}
cfg_path = ckpt_dir / 'run_config.json'
if cfg_path.is_file():
    previous = json.loads(cfg_path.read_text(encoding='utf-8'))
    if previous != RUN_CONFIG:
        differing = sorted(k for k in set(previous) | set(RUN_CONFIG)
                           if previous.get(k) != RUN_CONFIG.get(k))
        raise SystemExit(
            '已有分块是用另一套配置算的,不能混:' + chr(10)
            + ''.join(f'  {k}: 已有 {previous.get(k)!r} / 本次 {RUN_CONFIG.get(k)!r}' + chr(10)
                      for k in differing)
            + '  要么换回同样的运行时,要么删掉 blocks/ 整个重跑。')
else:
    cfg_path.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, sort_keys=True),
                        encoding='utf-8')
print('运行配置:', RUN_CONFIG['gpu'], '| fp16', USE_FP16)

def block_rows(i):
    return len(texts[i * BLOCK:(i + 1) * BLOCK])


def good(path, i):
    # A checkpoint counts as done only if it loads and has the right number of rows.
    # A runtime that dies mid-write leaves a truncated .npy. Its existence alone would
    # mark the block finished, and the failure would surface hours later at the merge
    # -- or not at all. Checking here costs a second and moves the failure to the start.
    try:
        return np.load(path).shape[0] == block_rows(i)
    except Exception:
        return False


done, dropped = set(), []
for path in ckpt_dir.glob('*.npy'):
    try:
        i = int(path.stem)
    except ValueError:
        continue
    if good(path, i):
        done.add(i)
    else:
        path.unlink()
        dropped.append(i)
if dropped:
    print(f'丢弃 {len(dropped)} 个损坏/写了一半的分块,将重算: {sorted(dropped)}')

todo = [i for i in range(n_blocks) if i not in done]
print(f'共 {n_blocks} 块 | 已完成 {len(done)} | 待跑 {len(todo)}')

oom_blocks = []
if todo:
    model = BGEM3FlagModel(str(model_path), use_fp16=USE_FP16, devices=DEVICE)
    print(f'配置 device={DEVICE} use_fp16={USE_FP16} batch_size={BATCH}')
    started = time.time()
    for k, i in enumerate(todo, 1):
        block = texts[i * BLOCK:(i + 1) * BLOCK]
        try:
            vecs = model.encode(block, batch_size=BATCH, max_length=MAX_LENGTH)['dense_vecs']
        except torch.cuda.OutOfMemoryError:
            oom_blocks.append(i)
            # A few blocks hold many long chunks at once. Retry those alone rather than
            # losing the run; batch size does not change a vector, only how many are in
            # flight, so the result is the same either way.
            print(f'  块 {i} 显存不足,降到 batch_size=2 重试')
            torch.cuda.empty_cache()
            vecs = model.encode(block, batch_size=2, max_length=MAX_LENGTH)['dense_vecs']
        # write beside the target then rename, so a death mid-write cannot leave a
        # half-written file under the real name
        tmp = ckpt_dir / f'{i}.npy.partial'
        # via a handle: np.save appends '.npy' to a path that lacks it, which would write
        # to a name this code never looks for and leave the rename with nothing to move
        with open(tmp, 'wb') as fh:
            np.save(fh, np.asarray(vecs, dtype=np.float32))
        tmp.replace(ckpt_dir / f'{i}.npy')
        elapsed = time.time() - started
        rate = elapsed / k
        print(f'  块 {i + 1}/{n_blocks} | 本次已跑 {k}/{len(todo)} | '
              f'{elapsed / 60:.1f} 分 | 预计还需 {rate * (len(todo) - k) / 60:.1f} 分')
    del model
    print('嵌入完成')
else:
    print('所有块都已完成,直接进入下一格')

## 7 · 合并,写出结果

把分块拼回一个矩阵,连同 row map 和 contract 一起写到 Drive。

row map 记的是 `(song_id, chunk_id, source_order)` —— PD-002 的替换规则要求保留这三个,
下游做去泄漏切分时要靠它把同一个 text component 的记录扣在一起。

In [ ]:
import numpy as np, json, datetime

parts = [np.load(ckpt_dir / f'{i}.npy') for i in range(n_blocks)]
E = np.concatenate(parts, axis=0)
assert E.shape[0] == len(rows), f'行数对不上: {E.shape[0]} vs {len(rows)}'
print('向量矩阵', E.shape, E.dtype)

np.save(OUT / 'repaired_corpus_v2_neutralised_bge_m3_embeddings.npy', E)

with (OUT / 'repaired_corpus_v2_neutralised_embedding_row_map.csv').open('w', encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    w.writerow(['row_index', 'song_id', 'chunk_id', 'source_order',
                'text_component_id', 'song_duplicate_group_id', 'song_component_weight'])
    for i, r in enumerate(rows):
        w.writerow([i, r['song_id'], r['chunk_id'], r['source_order'],
                    r['text_component_id'], r['song_duplicate_group_id'],
                    r['song_component_weight']])

contract = {
    'contract_version': 'chinese-rap-repaired-corpus-embeddings-v2-neutralised/1.0.0',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'corpus': {
        'content_sha256': EXPECTED_DIGEST,
        'chunks': len(rows),
        'songs': songs,
    },
    'model': {
        'name': 'BAAI/bge-m3',
        'weights_file': weights.name,
        'weights_sha256': WEIGHTS_SHA256,
    },
    # read back from the checkpoint directory, not from this session: the merge may run in a
    # later session than the one that produced most of the blocks, and reporting the current
    # session's device would assert a configuration that did not compute these vectors
    'configuration': {
        **json.loads((ckpt_dir / 'run_config.json').read_text(encoding='utf-8')),
        'batch_size': BATCH,
        'blocks_retried_at_batch_size_2_after_oom': sorted(oom_blocks),
        'order': 'corpus source order, not length-sorted',
        'single_configuration_enforced_by': (
            'run_config.json pinned to the checkpoint directory; a resume under any different '
            'device, precision, GPU, weights or corpus is refused rather than merged'
        ),
    },
    'embeddings': {
        'shape': list(E.shape),
        'dtype': str(E.dtype),
        'sha256': hashlib.sha256((OUT / 'repaired_corpus_v2_neutralised_bge_m3_embeddings.npy').read_bytes()).hexdigest(),
    },
    'why_a_single_run': (
        'Step 1 could not recover the historical configuration: all three candidates fell '
        'inside the 1e-4 cosine tolerance and the tolerance is about five times the spread '
        'between them. Embedding only the restored chunks would have put two unknown '
        'precisions in one retrieval space. One run removes the boundary rather than '
        'resolving it.'
    ),
    'not_comparable_to': (
        'canonical_clean_text_bge_m3_embeddings_v1.npy -- that vector set was produced under '
        'an unrecorded configuration which step 1 established cannot be identified. Metrics '
        'computed on v2 vectors must not be compared point-to-point against published v1 '
        'numbers; the corpus population differs as well.'
    ),
    'privacy': 'private -- vectors and row map must not be committed or published',
}
(OUT / 'repaired_corpus_v2_neutralised_embedding_contract.json').write_text(
    json.dumps(contract, ensure_ascii=False, indent=2, sort_keys=True) + chr(10),
    encoding='utf-8')

print()
for p in sorted(OUT.glob('*')):
    if p.is_file():
        print(f'  {p.name}  {p.stat().st_size / 1e6:.1f} MB')

## 8 · 结果

把下面这一格的输出**整个复制发给我**。里面没有歌词,只有配置和摘要。

In [ ]:
print(json.dumps({k: v for k, v in contract.items()
                  if k not in ('why_a_single_run', 'not_comparable_to')},
                 ensure_ascii=False, indent=2))

## 跑完之后

1. 把上一格的输出复制发我
2. Drive 上的 `embeddings-v2-neutralised/` **留着**
3. `embeddings-v2-neutralised/blocks/` 是断点文件，确认合并成功后可以删
4. **代码执行程序 → 断开连接并删除运行时**

向量出来之后我在本地跑"语义臂去主题"的对照，不用你再开 Colab。
